# 课后作业 · 多轮对话机器人（长度限制 + 摘要压缩）

## 作业要求

1. **对话长度限制**：对话历史最大 20 轮，对应存储的消息条目不超过 40 条，避免历史过长占用 Token、拖慢响应
2. **对话摘要压缩**：超过 20 轮时，调用额外的小参数大模型对早期对话做摘要，在保留核心信息的前提下减少 Token 占用，**替代直接截断**
3. **实现一个支持多轮对话的聊天机器人，优先使用 Redis 存储对话历史**

## 交付物

| 文件 | 说明 |
|---|---|
| `homework_memory_chat.py` | 完整实现：存储层 / 摘要层 / 记忆层 / 业务链 / CLI / 离线自测 |
| `homework_memory_chat.ipynb` | 本笔记：设计说明 + 逻辑验证（离线可跑） |

## 一句话思路

**不是把老消息删掉，而是把老消息「换一种更短的说法」继续带着走** —— 这正是摘要压缩和粗暴截断的本质区别。

## 分层设计

```mermaid
graph TD
    A["业务层<br/>RunnableWithMessageHistory"] -->|读写历史| B["记忆层<br/>SummarizingHistory"]
    B -->|超过 40 条| C["摘要层<br/>小参数模型"]
    B -->|读写 JSON| D["存储层<br/>RedisStore / MemoryStore"]
    C -->|摘要替换早期消息| B
```

四层的职责：

| 层 | 负责 | 换实现时的代价 |
|---|---|---|
| 业务层 | 组装 prompt、调模型、按 `session_id` 取历史 | 挂在 LangGraph 上时换掉 |
| 记忆层 | 判长度、触发压缩、拼出给模型的 `messages` | 不动 |
| 摘要层 | 把「早期对话」压成一段摘要 | 换模型只改 `SUMMARY_MODEL` |
| 存储层 | 会话历史落 Redis / 内存 | 换数据库只实现 `load/save/delete` |

Redis 里的数据结构（一个会话一个 key）：

```json
chat:history:<session_id> = {
  "summary": "用户叫张三，关心 Python 部署……",
  "messages": [ {"type": "human", "data": {...}}, {"type": "ai", "data": {...}} ]
}
```

`messages` 只留最近若干条，`summary` 承担更早的信息；带 TTL（默认 7 天）自动过期。

In [1]:
import sys
from pathlib import Path

# 让 notebook 无论在 week23/ 还是子目录打开都能 import 到作业模块
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "homework_memory_chat.py").exists())
sys.path.insert(0, str(root))

import homework_memory_chat as hw
from langchain_core.messages import AIMessage, HumanMessage

print("工作目录     :", Path.cwd())
print("模块文件     :", Path(hw.__file__).name)
print("最大轮数     :", hw.MAX_TURNS, "轮")
print("消息条数上限 :", hw.MAX_MESSAGES, "条")
print("压缩后保留   :", hw.KEEP_RECENT, "条最近消息")
print("摘要字数上限 :", hw.SUMMARY_MAX_CHARS, "字")
print("Redis 地址   :", hw.REDIS_URL)
print("摘要模型     :", "SUMMARY_MODEL（未配置则复用主模型）")

工作目录     : d:\project\step1\week23
模块文件     : homework_memory_chat.py
最大轮数     : 20 轮
消息条数上限 : 40 条
压缩后保留   : 20 条最近消息
摘要字数上限 : 300 字
Redis 地址   : redis://127.0.0.1:6379/0
摘要模型     : SUMMARY_MODEL（未配置则复用主模型）


## 1 · 存储层：Redis 优先，自动降级

两个实现共用 `HistoryStore` 接口（`load` / `save` / `delete`），存储的都是同一份 JSON 文档：

| 实现 | 特点 | 使用场景 |
|---|---|---|
| `RedisStore` | 进程重启不丢，多实例共享，带 TTL | 正式方案（作业要求） |
| `MemoryStore` | 存进程内存，接口完全一致 | 本机没 Redis 时的降级 / 单元测试 |

`create_store()` 会先尝试 `ping()` Redis，失败就打警告并降级 —— 这样作业在没有 Redis 的机器上也能跑通。

启动 Redis 的两种方式：

```bash
docker run -d --name redis-chat -p 6379:6379 redis:7-alpine
# 或者把 .env 里的 REDIS_URL 指向已有实例（例如课堂的 192.168.2.197:6379/13）
```

In [2]:
store = hw.create_store()          # 连不上 Redis 会自动降级
print("实际使用的存储：", type(store).__name__)

# 存一份再读回来，验证接口一致
store.save("demo-key", {"summary": "用户叫张三", "messages": []})
print("读回：", store.load("demo-key"))
store.delete("demo-key")
print("删除后：", store.load("demo-key"))

[存储] Redis 不可用（ConnectionError: Error 10061 connecting to 127.0.0.1:6379. 由于目标计算机积极拒绝，无法连接。.）
       → 降级为内存存储，重启进程后历史会丢
       → 想用 Redis：docker run -d --name redis-chat -p 6379:6379 redis:7-alpine
实际使用的存储： MemoryStore
读回： {'summary': '用户叫张三', 'messages': []}
删除后： {'summary': '', 'messages': []}


## 2 · 摘要层：用小参数模型压缩早期对话

摘要提示词的四条硬性要求（写在 `SUMMARY_PROMPT` 里）：

1. 保留用户的事实信息（姓名、身份、偏好、明确说过的约束）
2. 保留已确认的结论、未完成的任务、最新诉求
3. 省略寒暄、重复确认、客套话
4. 第三人称陈述，直接给结论，不加评论

**为什么要用小模型**：摘要只是「信息搬运」，不需要强推理；用便宜的小模型（或主模型的低 `max_tokens` 档）能把成本压到最低。

**多轮压缩怎么不丢信息**：每次压缩都会把「已有摘要」和「新增对话」一起喂进去，产出新摘要 —— 摘要滚雪球，而不是各自为政。

**失败兜底**：摘要模型超时/额度用尽时，退化为「截取早期对话尾部 N 字」，保证主对话流程不中断，历史也不会无限增长。

In [3]:
summarizer = hw.get_summarizer()
print("摘要模型类型：", type(summarizer).__name__)
print("摘要 max_tokens：", hw.SUMMARY_MAX_TOKENS)
print("摘要字数上限：", hw.SUMMARY_MAX_CHARS)
print("\n提示词模板：\n" + "-" * 50)
print(hw.SUMMARY_PROMPT)
print("-" * 50)
print("（本格只构造对象，不发起请求，不产生费用）")

摘要模型类型： ChatOpenAI
摘要 max_tokens： 400
摘要字数上限： 300

提示词模板：
--------------------------------------------------
你在为一段多轮对话做摘要压缩，摘要会替代原始对话继续参与后续推理。

请把下面内容压缩成不超过 {max_chars} 字的中文摘要：
1. 保留用户的事实信息（姓名、身份、偏好、明确说过的约束）
2. 保留已确认的结论，以及尚未完成的任务和用户最新的诉求
3. 省略寒暄、重复确认、客套话
4. 用第三人称陈述，直接给结论，不要评论

【已有摘要】
{previous_summary}

【新增对话】
{conversation}

只输出摘要正文，不要加标题或解释。
--------------------------------------------------
（本格只构造对象，不发起请求，不产生费用）


## 3 · 记忆层：长度限制怎么算

`SummarizingHistory` 实现 `BaseChatMessageHistory` 的三个成员，LangChain 就能直接用它管历史：

| 成员 | 作用 |
|---|---|
| `messages`（属性） | 读出历史；有摘要时在最前面插入一条 `SystemMessage("【早前对话摘要】…")` |
| `add_messages()` | 追加新消息，**超过 40 条就触发压缩** |
| `clear()` | 删除该会话（Redis `DEL`） |

长度口径：

- 1 轮 = 1 条 human + 1 条 ai = 2 条消息
- 上限 20 轮 → 40 条；追加后 `len > 40`（即第 21 轮）触发压缩
- 压缩后保留最近 20 条（10 轮）+ 1 条摘要，下次再超限才会再压缩 → **压缩是批量的，不是每轮都调模型**

⚠️ 关键点：被压缩掉的原始消息**不会**继续留在存储里，但摘要会随每次压缩滚动更新，所以 20 轮之前的信息仍以摘要形式参与后续推理。

In [4]:
import inspect

print(inspect.getsource(hw.SummarizingHistory.add_messages))
print(inspect.getsource(hw.SummarizingHistory._compress))

    def add_messages(self, messages) -> None:
        doc = self.store.load(self.session_id)
        doc["messages"].extend(message_to_dict(m) for m in messages)

        if len(doc["messages"]) > self.max_messages:
            doc = self._compress(doc)

        self.store.save(self.session_id, doc)

    def _compress(self, doc: dict) -> dict:
        raw = doc["messages"]
        old, recent = raw[: -self.keep_recent], raw[-self.keep_recent:]

        summary = self._summarize(doc["summary"], self._render(old))

        if self.verbose:
            print(
                f"\n[摘要压缩] session={self.session_id} "
                f"{len(raw)} 条 → 摘要({len(summary)} 字) + 最近 {len(recent)} 条"
            )

        return {"summary": summary, "messages": recent}



In [5]:
class FakeSummarizer:
    """假摘要器：只为验证逻辑，不调用任何模型。"""

    def __init__(self):
        self.calls = 0

    def invoke(self, prompt):
        self.calls += 1

        class _Resp:
            content = f"【摘要#{self.calls}】用户多次说明自己叫张三，助手已确认记住该信息。"

        return _Resp()


store = hw.MemoryStore()
fake = FakeSummarizer()
history = hw.SummarizingHistory("demo", store, fake)

for i in range(1, 26):
    history.add_messages([
        HumanMessage(content=f"第 {i} 轮：我的名字是张三，请记住。"),
        AIMessage(content=f"第 {i} 轮：好的，记住了。"),
    ])
    s = history.stats()
    print(
        f"第 {i:2d} 轮 | 存储 {s['stored_messages']:2d} 条 | "
        f"摘要 {'有' if s['has_summary'] else '无'} | 送模型 {s['prompt_messages']:2d} 条"
    )

final = history.stats()
print(f"\n摘要模型被调用 {fake.calls} 次；最终存储 {final['stored_messages']} 条（上限 {hw.MAX_MESSAGES}）")
print("送给模型的第一条消息：", history.messages[0].content)

第  1 轮 | 存储  2 条 | 摘要 无 | 送模型  2 条
第  2 轮 | 存储  4 条 | 摘要 无 | 送模型  4 条
第  3 轮 | 存储  6 条 | 摘要 无 | 送模型  6 条
第  4 轮 | 存储  8 条 | 摘要 无 | 送模型  8 条
第  5 轮 | 存储 10 条 | 摘要 无 | 送模型 10 条
第  6 轮 | 存储 12 条 | 摘要 无 | 送模型 12 条
第  7 轮 | 存储 14 条 | 摘要 无 | 送模型 14 条
第  8 轮 | 存储 16 条 | 摘要 无 | 送模型 16 条
第  9 轮 | 存储 18 条 | 摘要 无 | 送模型 18 条
第 10 轮 | 存储 20 条 | 摘要 无 | 送模型 20 条
第 11 轮 | 存储 22 条 | 摘要 无 | 送模型 22 条
第 12 轮 | 存储 24 条 | 摘要 无 | 送模型 24 条
第 13 轮 | 存储 26 条 | 摘要 无 | 送模型 26 条
第 14 轮 | 存储 28 条 | 摘要 无 | 送模型 28 条
第 15 轮 | 存储 30 条 | 摘要 无 | 送模型 30 条
第 16 轮 | 存储 32 条 | 摘要 无 | 送模型 32 条
第 17 轮 | 存储 34 条 | 摘要 无 | 送模型 34 条
第 18 轮 | 存储 36 条 | 摘要 无 | 送模型 36 条
第 19 轮 | 存储 38 条 | 摘要 无 | 送模型 38 条
第 20 轮 | 存储 40 条 | 摘要 无 | 送模型 40 条

[摘要压缩] session=demo 42 条 → 摘要(29 字) + 最近 20 条
第 21 轮 | 存储 20 条 | 摘要 有 | 送模型 21 条
第 22 轮 | 存储 22 条 | 摘要 有 | 送模型 23 条
第 23 轮 | 存储 24 条 | 摘要 有 | 送模型 25 条
第 24 轮 | 存储 26 条 | 摘要 有 | 送模型 27 条
第 25 轮 | 存储 28 条 | 摘要 有 | 送模型 29 条

摘要模型被调用 1 次；最终存储 28 条（上限 40）
送给模型的第一条消息： 【早前对话摘要】【摘要#1】用户多次说明自己叫张三，助手已确认记住该信息

## 4 · 结果解读

从上面的输出可以看到三个关键现象：

1. **第 1~20 轮**：存储条数 2→40 线性增长，没有摘要，送模型的条数就是全部历史
2. **第 21 轮**：42 条 → 触发压缩 → 存储降到 20 条，同时出现 1 条摘要；因此送模型是 21 条（摘要 + 20 条 + 本轮输入）
3. **之后每 10 轮**再压一次（20 + 10×2 = 40，第 11 轮超限），摘要模型调用次数远低于轮数

### 摘要 vs 直接截断

| 维度 | 直接截断（丢最早 N 条） | 摘要压缩（本实现） |
|---|---|---|
| 早期信息 | 彻底丢失，「我叫张三」这种关键事实没了 | 保留在摘要里，继续参与推理 |
| Token 占用 | 降到固定上限 | 同样降到上限（摘要 300 字封顶） |
| 额外成本 | 0 | 每 10 轮一次小模型调用 |
| 一致性 | 模型可能反复问已经问过的事 | 摘要明确了已知背景，不会重复确认 |

代价也很明确：**摘要是有损的**，细节（具体数字、代码片段）可能被压缩掉，所以提示词里要求「保留事实信息与未完成任务」。

In [6]:
RUN_DEMOS = False      # ← 改成 True 才会真正调用模型（会产生费用）

if RUN_DEMOS:
    store_real = hw.create_store()
    bot = hw.build_chatbot(store_real, hw.get_summarizer())

    for q in ["我叫张三，正在做 YOLOv8 的 ONNX 部署。", "我刚才说我叫什么？", "我在做什么项目？"]:
        print("你：", q)
        print("AI：", hw.ask(bot, q, session_id="nb-demo"))
        print("-" * 60)
else:
    print("（RUN_DEMOS=False，跳过真实调用）真实模式用法：")
    print('  bot = hw.build_chatbot(hw.create_store(), hw.get_summarizer())')
    print('  hw.ask(bot, "我叫什么？", session_id="nb-demo")')
    print("\n命令行版本更直接：python homework_memory_chat.py")

（RUN_DEMOS=False，跳过真实调用）真实模式用法：
  bot = hw.build_chatbot(hw.create_store(), hw.get_summarizer())
  hw.ask(bot, "我叫什么？", session_id="nb-demo")

命令行版本更直接：python homework_memory_chat.py


## 5 · 可调参数（都能在 `.env` 覆盖）

| 变量 | 默认 | 作用 |
|---|---|---|
| `CHAT_MAX_TURNS` | `20` | 最大对话轮数，`MAX_MESSAGES = 2 × 轮数` |
| `CHAT_KEEP_RECENT` | `20` | 压缩后保留的最近消息条数（= 10 轮原始上下文） |
| `SUMMARY_MAX_CHARS` | `300` | 摘要字数上限，写进提示词 |
| `SUMMARY_MAX_TOKENS` | `400` | 摘要模型输出上限，防止摘要本身失控 |
| `SUMMARY_MODEL` | 未设（复用主模型） | 指定专用小模型，例如更便宜的 flash 档 |
| `REDIS_URL` | `redis://127.0.0.1:6379/0` | Redis 连接串 |
| `CHAT_SESSION_TTL` | `604800`（7 天） | 会话过期时间，避免 Redis 无限增长 |
| `CHAT_SESSION_ID` | `homework-demo` | CLI 默认会话 id |

**调参建议**：`KEEP_RECENT` 太小会导致上下文断裂、压缩过于频繁；太大则压缩间隔变长、单次 prompt 变大。经验值是「保留量 ≈ 上限的一半」。

## 6 · 提交与验收

### 运行方式

```bash
# 离线自测（不花 token，验证长度限制与压缩）
python homework_memory_chat.py --demo

# 真实对话（Redis 可用则用 Redis，否则自动降级内存）
python homework_memory_chat.py
```

### 三条要求逐条对照

| 要求 | 实现位置 | 验证方式 |
|---|---|---|
| 最多 20 轮 / 40 条 | `MAX_TURNS` / `MAX_MESSAGES`，`add_messages` 里判断 | 本笔记第 4 节：第 20 轮正好 40 条，第 21 轮压缩 |
| 超限时用摘要替代截断 | `_compress()` + `_summarize()`，小模型按 `SUMMARY_MODEL` | 输出里出现 `[摘要压缩] 42 条 → 摘要 + 最近 20 条` |
| 多轮聊天机器人 + Redis 优先 | `build_chatbot()` + `RedisStore`（`create_store` 优先尝试） | 装上 Redis 后 key `chat:history:<session>` 可见 |

### 两点说明

1. **`RunnableWithMessageHistory` 在 langchain 1.4 起被标记为 deprecated**，官方推荐改用 LangGraph 的 checkpointer。作业按课堂口径实现；如果要迁移，只需替换业务层，`SummarizingHistory` 这套「限制 + 摘要」逻辑可原样复用（LangGraph 也接受自定义的 `BaseChatMessageHistory`）。
2. **成本控制**：本笔记所有会调模型的格子都在 `RUN_DEMOS` 之后；摘要调用次数是「每超出一次压缩一次」，25 轮只调用 1 次。

### 延伸

- 摘要只压「早期对话」，最近的 10 轮始终是原始消息，保证刚刚说过的话不会失真
- 想更进一步：把摘要改成「结构化字段」（用户档案：姓名/目标/约束），比自然语言摘要更省 token 也更好用